<a href="https://colab.research.google.com/github/giovannidipace6-ops/Vector-RAG/blob/main/Vector_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Fase 1: Ingestione
```Documents --> Chunking --> Embedding --> Vector DB```

Fase 2: Recupero
```Query --> Embedding --> Search on Vector DB --> Retrieve k most relevant chunks```

Fase 3: Generazione```Query + Retrieved Context --> LLM --> Grounded Generation```

In [ ]:
!pip install langchain-community sentence-transformers

# install requirements

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 86.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 92.4 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
# carico i documenti della knowldedge base e mostro un minimo di contenuto dei primi due
docs_path = "/content/drive/MyDrive/docs"
loader = DirectoryLoader(
        path=docs_path,
        glob="*.txt",
        loader_cls=TextLoader
    )
print(f"Ok, loading documents from {docs_path} and i show you first two")
documents = loader.load()

for i, doc in enumerate(documents[:2]): # Mostro primi due documenti
        print(f"\nDocument {i+1}:")
        print(f"  Source: {doc.metadata['source']}")
        print(f"  Content length: {len(doc.page_content)} characters")
        print(f"  Content preview: {doc.page_content[:100]}...")
        print(f"  metadata: {doc.metadata}")

/tmp/ipykernel_3910/1892288527.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


Ok, loading documents from /content/drive/MyDrive/docs and i show you first two

Document 1:
  Source: /content/drive/MyDrive/docs/Google.txt
  Content length: 232201 characters
  Content preview: ﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and t...
  metadata: {'source': '/content/drive/MyDrive/docs/Google.txt'}

Document 2:
  Source: /content/drive/MyDrive/docs/Microsoft.txt
  Content length: 201014 characters
  Content preview: ﻿Microsoft
Microsoft Corporation is an American multinational Microsoft Corporation
corporation and ...
  metadata: {'source': '/content/drive/MyDrive/docs/Microsoft.txt'}


In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
# 2 Spacchetto i documenti in blocchi (chunks)
print("Splitting documents into chunks...")
"""
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
    )
chunks = text_splitter.split_documents(documents)
"""
# RecursiveCharacterTextSplitter spacchetta il testo a seguito di una ricerca ricorsiva dei caratteri nel testo, il risultato è una suddivisione per paragrfi sezioni o frasi (più naturale)
# Con Character TextSplitter si ottengono 1330 chunks con RecursiveCharacterTextSplitter 1356 (overlap 0) 1392 (overlap 200) - per entrambi i metodi
# Di seguito voglio provare a portare avanti un chunking con size 512 e overlap 50 --> 2996 chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100,
    separators = ["\n\n", "\n", ".", " "]
)
chunks = text_splitter.split_documents(documents)
if chunks:
   for i, chunk in enumerate(chunks[:5]):

    print(f"\n--- Chunk {i+1} ---")
    print(f"Source: {chunk.metadata["source"]}")
    print(f"Lenght: {len(chunk.page_content)} characters")
    print(f"Content:")
    print(chunk.page_content)
    print("-" * 50)

if len(chunks) > 5:
  print(f"\n ...and {len(chunks) - 5} more chunks")

Splitting documents into chunks...

--- Chunk 1 ---
Source: /content/drive/MyDrive/docs/Google.txt
Lenght: 600 characters
Content:
﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and technology
company focusing on online advertising, search engine
technology, cloud computing, computer software,
quantum computing, e-commerce, consumer
electronics, and artificial intelligence (AI).[9] It has
been referred to as "the most powerful company in the The Google logo used since 2015
world" by the BBC[10] and is one of the world's most
valuable brands.[11][12][13] Google's parent company,
Alphabet Inc., is one of the five Big Tech companies
alongside Amazon, Apple, Meta, and Microsoft.
--------------------------------------------------

--- Chunk 2 ---
Source: /content/drive/MyDrive/docs/Google.txt
Lenght: 867 characters
Content:
Google was founded on September 4, 1998, by
American computer scientists Larry Page and Sergey
Brin. Together, they own a

In [ ]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 104.6 MB/s  0:00:01


In [ ]:
from sentence_transformers import SentenceTransformer

# 3 Embedding e memorizzo nel vector store
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Effettuo l'embedding dei chunks\n")
text = [chunk.page_content for chunk in chunks]
#for t in range(2):
#  print(f" Chunk {t}:\n{text[t]}")
embeddings = model.encode(text, show_progress_bar=True)
print(len(embeddings))
dimension = embeddings.shape[1]
print(f"Embedding effettuato, ogni vettore ha dimensione {dimension} e i vettori sono {len(embeddings)}\n")

#print(text[0])
#print(embeddings[0])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Effettuo l'embedding dei chunks



Batches:   0%|          | 0/43 [00:00<?, ?it/s]

1365
Embedding effettuato, ogni vettore ha dimensione 384 e i vettori sono 1365



In [ ]:
import faiss
import numpy as np
# 4 Memorizzo i vettori nel vector store usando faiss indexFlatL2
index = faiss.IndexFlatL2(dimension) # creo l'indice
index.add(np.array(embeddings)) # aggiungo i vettori di embeddings all'indice
print(f"\n --- L'indice contiene {index.ntotal} vettori ---")


 --- L'indice contiene 1365 vettori ---


In [ ]:
# 5 Embedding della query
query = "How much did Microsoft pay to acquire GitHub?"




In [ ]:
# 5-6 Embedding della query e recupero dei k chunks più rilevanti
def retrieve(query, k):
  query_embedding = model.encode([query])
  distances, indices = index.search(query_embedding, k)
  #print(distances[0], indices[0])
  #distanza=distances[0]
  #print(distanza[0])

  return [text[i] for i in indices[0]]

#print(str(retrieve("Who is Elon Musk?")))

In [ ]:
!pip install ollama

In [ ]:
# 1. Installazione di zstd, pciutils ed Ollama
!sudo apt update && sudo apt install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Avvio del server Ollama in background tramite subprocess
import subprocess
import time
import requests

print("Avvio del server Ollama in corso...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Attendiamo che il server sia pronto
for _ in range(10):
    try:
        response = requests.get("http://localhost:11434/")
        if response.status_code == 200:
            print("✅ Ollama è ora attivo in background!")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    print("❌ Timeout: non è stato possibile avviare Ollama.")

# 3. Download del modello (Sostituisci llama3 con un altro modello se preferisci)
print("Download del modello in corso...")
!ollama pull llama3

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,000 kB]
Get:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble InRelease
Get:8 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.0 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [75.7 kB]
Get:13 http://secu

In [ ]:
from ollama import chat
!ollama pull llama3
# funzione per passare la query alla funzione retrieve che effettuerà embedding e ricerka dei k chunks più rilevanti
def rag_query(question):
  context = retrieve(question,6)

  # Preparo il prompt per il modello
  prompt = f"""You are an useful assistant.
  Answer the question based only on this context:
  Context:
  {context}

  Question: {question}

  Answer:"""

  response = chat(
      model='llama3',
      messages=[{'role': 'user', 'content': prompt}],
  )

  return response.message.content

In [ ]:
# 7 Generazione della risposta finale
query = "What was the original name of Microsoft before it became Microsoft?"


print(rag_query(query))



"""
Dati per la valutazione, estrapolo una serie di query a partire dai documenti
Per ciascuna domanda poi ricavo la risposta desiderata dai documenti come expected_output
--- 1 ---
Source: Google began in January 1996 as a research project by Larry
Page and Sergey Brin while they were both PhD students at

q1 = "Who founded Google?"
expected_output = "Google was founded by Larry Page Sergey Brin"
actual_output = "According to the context, Google was founded by Sergey Brin and Larry Page, as stated in the article "Google Founded By Sergey Brin, Larry Page... And Hubert Chang??""

--- 2 ---
Source: Nvidia's first graphics accelerator, the NV1, was designed to process quadrilateral primitives (forward
texture mapping), a feature that set it apart from competitors, who preferred triangle primitives.[26]

q2 = "What was NVIDIA's first graphics accelerator called?"
expected_output = "The NVIDIA's first graphics accelerator was called NV1"
actual_output = "NVIDIA's first graphics accelerator was called the NV1."

--- 3 ---
Source: An Altair 8800 computer (left) with the marketing it as Altair BASIC.[10]: 108, 112–114  Gates and Allen
popular Model 33 ASR Teletype as
terminal, paper tape reader, and paper established Microsoft on April 4, 1975, with Gates as
tape punch CEO,[14] and Allen suggested the name "Micro-Soft", short

for micro-computer software.[15][16] In August 1977, the
company formed

q3 = "What was the original name of Microsoft before it became Microsoft?"
expected_output = "Original name of Microsoft was Micro-Soft"
actual_output = "Based on the provided context, there is no mention of Microsoft having an original name before it became Microsoft. The text only mentions the founding of Microsoft by Bill Gates and Paul Allen in 1975 to develop and sell BASIC interpreters for the Altair 8800, without mentioning a previous name."


"""

# 1. "What was NVIDIA's first graphics accelerator called?"
# 2. "Which company did NVIDIA acquire to enter the mobile processor market?"
# 3. "What was Microsoft's first hardware product release?"
# 4. "How much did Microsoft pay to acquire GitHub?"
# 5. "In what year did Tesla begin production of the Roadster?"
# 6. "Who succeeded Ze'ev Drori as CEO in October 2008?"
# 7. "What was the name of the autonomous spaceport drone ship that achieved the first successful sea landing?"
# 8. "What was the original name of Microsoft before it became Microsoft?"

According to the context, the original name of Microsoft was "Micro-Soft", which was suggested by Paul Allen.


'\nDati per la valutazione, estrapolo una serie di query a partire dai documenti\nPer ciascuna domanda poi ricavo la risposta desiderata dai documenti come expected_output\n--- 1 ---\nSource: Google began in January 1996 as a research project by Larry\nPage and Sergey Brin while they were both PhD students at\n\nq1 = "Who founded Google?"\nexpected_output = "Google was founded by Larry Page Sergey Brin"\nactual_output = "According to the context, Google was founded by Sergey Brin and Larry Page, as stated in the article "Google Founded By Sergey Brin, Larry Page... And Hubert Chang??""\n\n--- 2 ---\nSource: Nvidia\'s first graphics accelerator, the NV1, was designed to process quadrilateral primitives (forward\ntexture mapping), a feature that set it apart from competitors, who preferred triangle primitives.[26]\n\nq2 = "What was NVIDIA\'s first graphics accelerator called?"\nexpected_output = "The NVIDIA\'s first graphics accelerator was called NV1"\nactual_output = "NVIDIA\'s first g

In [ ]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.0 MB/s  0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.5.0
    Uninstalling click-8.5.0:
      Successfully uninstalled click-8.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [deepeval]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


In [ ]:
"""import os
import getpass

open_ai_key = "sk-ijklmnopabcd5678ijklmnopabcd5678ijklmnop"
os.environ['OPENAI_API_KEY'] = open_ai_key"""

'import os\nimport getpass\n\nopen_ai_key = "sk-ijklmnopabcd5678ijklmnopabcd5678ijklmnop"\nos.environ[\'OPENAI_API_KEY\'] = open_ai_key'

ContextualPrecisionMetric: valuta se il reranker adottato riordina i nodi più rilevanti più in alto rispetto ai nodi meno rilevanti

ContextualRecallMetric:valuta se il modello di embedding usato cattura accuratamente e recupera le informazioni rilevanti basandosi sul contesto dell'input

ContextualRelevancyMetric: valuta se la dimensione del chunk e il recupero dei top-K sono tali da recuperare l'informazione senza troppi recuperi irrilevanti

In [ ]:
from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric
)
from deepeval.models import OllamaModel

eval_model = OllamaModel(
    model="llama3",
)
contextual_precision = ContextualPrecisionMetric(model=eval_model)
contextual_recall = ContextualRecallMetric(model=eval_model)
contextual_relevancy = ContextualRelevancyMetric(model=eval_model)

In [ ]:
from deepeval.test_case import LLMTestCase
test_case = LLMTestCase(
    input="What was the original name of Microsoft before it became Microsoft?",
    actual_output='Original name of Microsoft was Micro-Soft',
    expected_output="Original name of Microsoft was Micro-Soft",
    retrieval_context=[
        """An Altair 8800 computer (left) with the marketing it as Altair BASIC.[10]: 108, 112–114  Gates and Allen
popular Model 33 ASR Teletype as
terminal, paper tape reader, and paper established Microsoft on April 4, 1975, with Gates as
tape punch CEO,[14] and Allen suggested the name "Micro-Soft", short

for micro-computer software.[15][16] In August 1977, the
company formed"""
    ]
)

In [ ]:
contextual_precision.measure(test_case)
print("Score: ", contextual_precision.score)
print("Reason: ", contextual_precision.reason)

contextual_recall.measure(test_case)
print("Score: ", contextual_recall.score)
print("Reason: ", contextual_recall.reason)

contextual_relevancy.measure(test_case)
print("Score: ", contextual_relevancy.score)
print("Reason: ", contextual_relevancy.reason)

Output()

Output()

Score:  1.0
Reason:  The score is 1.00 because all nodes are correctly ranked, with the first two 'yes' verdicts being the most relevant to the input question about the original name of Microsoft, and the third 'no' verdict being correctly ranked lower due to its lack of direct connection to the question, specifically the mention of the Altair 8800 computer and the marketing of Altair BASIC.


Output()

Score:  1.0
Reason:  The score is 1.00 because the expected output perfectly matches the information in the retrieval context, with the original name of Microsoft being attributed to the correct node.


Score:  0.5
Reason:  The score is 0.50 because the retrieval context provides information about the founding of Microsoft, which is indirectly related to the original name of Microsoft, but the specific statement about the name 'Micro-Soft' being suggested is the most relevant part, making the score moderate.


In [ ]:
#!pip install ChatVertexAI

In [ ]:
"""!pip install -U -q "ragas==0.4.3" "langchain-core<1.4.0" "langchain-community<0.4.2" "langchain-google-vertexai: 3.2.3" "langchain: 1.3.1"

ragas: 0.4.3
langchain-community: 0.4.2
langchain-google-vertexai: 3.2.3
langchain-core: 1.4.0
langchain: 1.3.1

from datasets import Dataset
import os
from ragas import evaluate
from ragas.metrics import faithfulness, answer_correctness"""

# Ragas parrebbe avere dei problemi di versionamenti abbastanza difficili da risolvere
